In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os
import ast
from collections import Counter


In [2]:
df = pd.read_csv("Threshold_for_low_ligands.csv")


In [3]:
df.head(30)

,feature,best_threshold,non_zero_before,zero_before,non_zero_after,zero_after,max_separation,percent_non_zero_after,percent_zero_after
0,model_dipolemoment_vburminconf,1.498223,0,11,22,4,29,100.000000,26.666667
1,model_vbur_far_vtot_boltz,28.522257,2,13,20,2,29,90.909091,13.333333
2,model_volume_boltz,453.276269,2,13,20,2,29,90.909091,13.333333
3,model_vbur_far_vbur_boltz,5.174745,2,12,20,3,27,90.909091,20.000000
4,model_vbur_qvbur_max_max,28.266858,2,12,20,3,27,90.909091,20.000000
5,model_sterimol_L_boltz,7.493029,2,11,20,4,25,90.909091,26.666667
6,model_surface_area_boltz,350.436274,2,11,20,4,25,90.909091,26.666667
7,model_vbur_far_vtot_max,33.275170,2,11,20,4,25,90.909091,26.666667
8,model_vbur_ovtot_max_vburminconf,143.106781,3,15,19,0,31,86.363636,0.000000
9,model_vbur_qvtot_max_max,176.668689,3,15,19,0,31,86.363636,0.000000


In [4]:
ligand = pd.read_csv('High_ligands_final_final_features_output.csv')


ligand['index_column'] = ligand.index

ligand.to_csv('High_ligands_final_final_features_output_index.csv', index=False)


In [5]:
ligand.head(5)

,input,model_E_oxidation_boltz,model_E_reduction_boltz,model_E_solv_cds_boltz,model_E_solv_elstat_boltz,model_E_solv_total_boltz,model_Pint_P_int_boltz,model_Pint_P_max_boltz,model_Pint_P_min_boltz,model_Pint_dP_boltz,...,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,yield,SMILES,index_column
0,0000000000000000000000000000000001000000000101...,0.256783,0.023705,-4.816398,-9.268672,-15.162958,20.015041,36.363710,12.033858,3.981717,...,102.835390,82.80130,77.535370,422.66220,1.811474,-0.065044,532.77014,91,COC1=CC=C(N2C(P(C(C)(C)C)C(C)(C)C)=CC=N2)C(OC)=C1,0
1,0010100000000000000000000000000001000000000101...,0.263688,0.020393,-9.382906,-7.951180,-15.298017,20.243896,40.929077,12.579647,4.573813,...,106.617520,61.98695,59.169624,549.79650,1.802640,-0.063585,650.82434,46,COC1=CC(OC)=C(C(C)(C)C)C=C1N2C(P(C3CCCCC3)C4CC...,1
2,0000000000000000000000000000000001000000000101...,0.260458,0.020929,-6.768596,-8.390507,-15.078186,20.086765,39.388270,12.579207,4.014885,...,101.890366,82.45664,77.446250,505.30374,1.811892,-0.063683,664.99713,93,CC(P(C1=CC=NN1C2=CC(C(C)(C)C)=C(OC)C=C2OC)C(C)...,2
3,0010100000000000000000000000000101000000000001...,0.255795,0.022808,-9.089910,-8.239961,-16.703497,20.002974,38.729824,12.072245,4.542177,...,121.368340,56.29661,56.541195,437.52540,1.803918,-0.065115,538.28253,33,COC1=C(C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2)C=CC(O...,3
4,0010100000000001000000000000000101000000000001...,0.250412,0.023673,-9.482734,-7.935542,-17.410470,20.461084,38.761520,13.001906,4.536157,...,116.998710,77.85579,77.615710,479.07794,1.788780,-0.067359,584.05963,44,CN(C1=C(C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2)C(OC)...,4


In [6]:
cols = ligand.columns

In [10]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

output_folder = "High_ligands_final_final_output_ligand_plot_thresholding"
os.makedirs(output_folder, exist_ok=True)

feature_indices = []

for feature in df['feature']:
    
    threshold = df.loc[df['feature'] == feature, 'best_threshold'].values[0]
    feature_column = ligand[feature].values
    yield_column = ligand['yield'].values  
    input_column = ligand['SMILES'].values  
    indices_for_feature = []
    
    for i in range(len(feature_column)):
        if feature_column[i] >= threshold and yield_column[i] == 0:
            indices_for_feature.append(i)  # Store the index of the row
    if indices_for_feature:
        feature_indices.append([feature, indices_for_feature])  # Store feature name and indices
    before_threshold = ligand[feature_column < threshold]
    after_threshold = ligand[feature_column >= threshold]
    
    # Plotting
    plt.figure(figsize=(10, 6))
    
    # Set the marker size (s) to make the circles bigger
    plt.scatter(before_threshold[feature], before_threshold['yield'], 
                color='blue', label='Before Threshold', alpha=0.6, edgecolors='k', s=100)  # Increase 's' for bigger markers
    plt.scatter(after_threshold[feature], after_threshold['yield'], 
                color='red', label='After Threshold', alpha=0.6, edgecolors='k', s=100)  # Increase 's' for bigger markers
    
    # Add threshold line
    plt.axvline(threshold, color='black', linestyle='--', label=f'Threshold = {threshold}')

    # Set title and labels with bold text
    plt.title(f'Scatter Plot of {feature} vs Yield (Threshold = {threshold})', fontweight='bold')
    plt.xlabel(f'{feature} Value', fontweight='bold')
    plt.ylabel('Yield Value', fontweight='bold')
    
    # Set legend inside the plot area (use 'loc' for positioning)
    legend_font = font_manager.FontProperties(weight='bold', size=12)  # Increase font size
    plt.legend(prop=legend_font)  # Legend inside the plot, upper-left corner

    # Set y-limit to -1 to 100
    plt.ylim(-1, 100)
    
    # Add grid to the plot
    plt.grid(True, linestyle='--', alpha=0.7)  # Add grid with dashed lines and slight transparency
    
    # Increase the resolution of the plot when saving it
    plt.savefig(os.path.join(output_folder, f'{feature}_scatter_plot.png'), dpi=300, bbox_inches='tight')  # Higher DPI for better quality
    plt.close()  # Close the plot to avoid display issues

# Create a DataFrame from the list of feature names and indices
feature_indices_df = pd.DataFrame(feature_indices, columns=['feature', 'indices'])

# Save the DataFrame to a CSV file (optional)
feature_indices_df.to_csv(os.path.join(output_folder, 'feature_indices.csv'), index=False)

# Display the DataFrame
print("Feature and corresponding indices DataFrame:")
print(feature_indices_df)


Feature and corresponding indices DataFrame:
Empty DataFrame
Columns: [feature, indices]
Index: []


In [29]:
len(df.sort_values(by='percent_non_zero_after', ascending=False))

190

In [30]:
df1 = df
sorted_df1 = df1.sort_values(by='percent_non_zero_after', ascending=False)
top_10_features = sorted_df1.head(10)['feature'].tolist()

df2 = feature_indices_df

top_features = df2[df2['feature'].isin(top_10_features)]

top_features.to_csv('Low_yield_ligands_final_top_features_indices.csv', index=False)


In [31]:
top_features.head()

,feature,indices


In [14]:
df = top_features

df.loc[:, 'indices'] = df['indices'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

value_row_count = Counter()

for index, row in df.iterrows():
    unique_values = set(row['indices']) 
    for value in unique_values:
        value_row_count[value] += 1 

top_20_values = value_row_count.most_common(20)

top_20_df = pd.DataFrame(top_20_values, columns=['Value', 'Rows_Appeared_In'])


print("Top 20 values and the number of rows they appear in:")
print(top_20_df)

top_20_df.to_csv('Low_yield_ligands_top_20_analysis.csv', index=False)


Top 20 values and the number of rows they appear in:
    Value  Rows_Appeared_In
0      13                10
1      14                10
2      15                10
3       8                 9
4       4                 9
5      10                 9
6      17                 9
7      18                 9
8       5                 8
9       7                 8
10     16                 8
11      3                 8
12      0                 5
13      2                 4
14      1                 4


In [16]:
df.loc[:, 'indices'] = df['indices'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

top_20_values_set = set([value for value, _ in top_20_values])  

df.loc[:, 'top_20_count'] = df['indices'].apply(
    lambda x: len(set(x) & top_20_values_set)  )

print(df[['feature', 'top_20_count']])

df.to_csv('Low_yield_ligands_final_ligand_updated_with_top_20_count.csv', index=False)


                             feature  top_20_count
0     model_dipolemoment_vburminconf             4
18             model_sterimol_B5_min            12
20          model_vbur_near_vbur_min            12
21  model_vbur_near_vbur_vburminconf            12
22          model_nbo_bd_e_max_boltz            13
23               model_pyr_alpha_min            13
24             model_sterimol_B1_max            13
25        model_vbur_qvbur_min_boltz            13
26         model_nbo_bds_e_min_boltz            14
27        model_vbur_ovbur_min_delta            14
